# Python for AI — Class 21
### Neural Networks with PyTorch: build the network from the slides

In the slides we learned every piece of a neural network. Today we **build the same network in PyTorch** and watch it learn.

This is **the map** from the slides:

```
   INPUT LAYER            HIDDEN LAYER                 OUTPUT LAYER
   (the facts)        (3 neurons: +b, then ReLU)        (1 neuron: +b)

  study_hours ──┬────▶ ( neuron 1 ) ──┐
                ├────▶ ( neuron 2 ) ──┼────▶ ( output ) ──▶ predicted Maths mark
  phone_hours ──┴────▶ ( neuron 3 ) ──┘
```

| Part | What we do |
|------|------------|
| 🧑‍🏫 **Class Problem, part A** | Build the map in PyTorch and **look inside**: weights, bias, ReLU, forward pass, loss, backprop |
| 🧑‍🏫 **Class Problem, part B** | **Train it properly** and use it for a new student |
| 🧑‍🏫 **Example 2** | A network that **reads handwritten digits** (0 to 9) |
| 🏠 **Home Problem** | **You** do the same steps for **car prices** |

**Using Colab?** Click the **folder icon** on the left → **upload** `students_100.csv` and `cars.csv` first. (The digits come inside sklearn, no file needed.)

In [ ]:
# On your own laptop, install once:  pip install torch scikit-learn
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

torch.manual_seed(42)        # the same "random" starting weights for everyone

# 🧑‍🏫 Class Problem: guess a student's Maths mark

**The problem:** a school wants to guess a student's **Maths mark** from 2 facts: **study hours** and **phone hours**.

We solve it **together**, step by step.

## Part A: Look inside the network

### Step 1: The inputs

**Slide reminder:** inputs are the **facts** we give the network. One input = one column.

Networks want the data like a table: **one row per student**, **one column per input**.

In [ ]:
students = pd.read_csv("students_100.csv")
print(students[["study_hours", "phone_hours", "maths"]].head())

X = torch.tensor(students[["study_hours", "phone_hours"]].values, dtype=torch.float32)   # the 2 inputs
y = torch.tensor(students[["maths"]].values, dtype=torch.float32)                         # the real marks
print("X shape:", X.shape, "  y shape:", y.shape)

`torch.Size([100, 2])` = **100 students**, **2 inputs** each. `y` has 1 column: the real Maths mark.

`dtype=torch.float32` = decimal numbers. Neural networks always use decimals.

### Step 2: Build the map

**Slide reminder:**
- `nn.Linear(2, 3)` = the **lines** (weights) + the **+b** (biases) from 2 inputs to 3 hidden neurons.
- `nn.ReLU()` = the **ReLU** tag on each hidden neuron (negative → 0).
- `nn.Linear(3, 1)` = the lines + bias from 3 hidden neurons to 1 output.
- `nn.Sequential(...)` = put them **in a row**, left → right.

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 3),     # input layer → hidden layer (6 weights + 3 biases)
    nn.ReLU(),           # activation
    nn.Linear(3, 1),     # hidden layer → output (3 weights + 1 bias)
)
print(model)
print("Weights + biases to learn:", sum(p.numel() for p in model.parameters()))

**13 numbers to learn:** 2 × 3 = 6 weights + 3 biases, then 3 × 1 = 3 weights + 1 bias.

Let's **see** them. Right now they are **random**: the network knows nothing yet.

In [ ]:
print("Hidden layer weights (one row per neuron):")
print(model[0].weight)
print("Hidden layer biases (+b):", model[0].bias)

### Step 3: The forward pass, with the numbers from the slides

On the slides, the student with **3 study hours** and **2 phone hours** got the guess **8.7**.

Let's put **the same weights and biases** from the slides into our PyTorch network and check.
(`torch.no_grad()` = "we are only setting numbers, not learning".)

In [ ]:
with torch.no_grad():
    model[0].weight[:] = torch.tensor([[ 0.5, -0.6],     # neuron 1: weight for study, weight for phone
                                       [-1.0,  0.4],     # neuron 2
                                       [ 0.8,  0.2]])    # neuron 3
    model[0].bias[:]   = torch.tensor([0.1, 0.2, -0.5])  # +b of each hidden neuron
    model[2].weight[:] = torch.tensor([[2.0, 1.0, 3.0]]) # output: one weight per hidden neuron
    model[2].bias[:]   = 1.0                             # +b of the output neuron

student = torch.tensor([[3.0, 2.0]])          # 3 study hours, 2 phone hours

hidden = model[1](model[0](student))          # layer 1, then ReLU
print("Hidden neurons after ReLU:", hidden)
print("Output (the guess):       ", model(student))

**The same as the slides!** Hidden neurons `0.4, 0, 2.3` (neuron 2 was −2.0, so ReLU **switched it off**), and the guess is **8.7**.

But the real mark for a student like this is about **55**. A very bad guess, because these weights were not **learned**.

### ✏️ QUICK CHECK 1: Your own forward pass

A student studies **5 hours** and uses the phone **1 hour**.

1. Make the tensor for this student (like `student` above).
2. Print the hidden neurons after ReLU, and the output.

Which hidden neuron is switched off (0)?

In [ ]:
# Write your code here

### Step 4: The loss: how wrong is the network?

**Slide reminder:** loss = all the errors as **one number**. `nn.MSELoss()` squares each error and takes the average. **Smaller = better.**

Now we guess for **all 100 students** at once.

In [ ]:
loss_fn = nn.MSELoss()

guesses = model(X)
loss = loss_fn(guesses, y)
print("First 3 guesses:", guesses[:3].flatten())
print("First 3 real:   ", y[:3].flatten())
print("Loss:", round(loss.item()))

The guesses are around 10, the real marks are around 50 → a **big** loss.

### Step 5: Backprop and one step of gradient descent

**Slide reminder:**
- `loss.backward()` = **backpropagation**: the blame goes back, and every weight gets its **gradient** (which way to change).
- `optimizer.step()` = **gradient descent**: every weight takes **one small step** downhill.
- `lr` = the **learning rate**: the size of the step.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

optimizer.zero_grad()        # clear old blame
loss = loss_fn(model(X), y)  # guess + how wrong
loss.backward()              # backprop: find the blame for every weight

print("Blame (gradient) for the output weights:", model[2].weight.grad)
print("Loss before the step:", round(loss.item()))

optimizer.step()             # one small step downhill
print("Loss after the step: ", round(loss_fn(model(X), y).item()))

**One step**, and the loss is already **much smaller**. That is learning!

The gradients are **negative** → PyTorch says "make these weights **bigger**", because our guesses are too small.

### ✏️ QUICK CHECK 2: A learning rate that is too big

Run the cell below **two times**: it takes one step with `lr=0.01` (10 times bigger).

What happens to the loss? Why? (Think of the hill in the fog from the slides.)

In [ ]:
big = torch.optim.SGD(model.parameters(), lr=0.01)
big.zero_grad()
loss = loss_fn(model(X), y)
loss.backward()
big.step()
print("Loss after a big step:", round(loss_fn(model(X), y).item()))

**Answer:** the loss gets **much bigger**, not smaller. The step is so big that it **jumps over** the bottom of the hill, to a higher place on the other side. That's why we choose a small learning rate.

## Part B: Train it properly

Now we do it **the real way**: split the data, train for many epochs, check the mock exam, and use the model.

### Step 6: Split: train / validation / test

**Slide reminder:** train = **homework** (60%), validation = **mock exam** (20%), test = **final exam** (20%, only once at the end).

We split **two times**: first take out the test part, then split the rest.

In [ ]:
X_np = students[["study_hours", "phone_hours"]].values
y_np = students["maths"].values

X_rest, X_test, y_rest, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)   # 25% of 80% = 20%
print("train:", len(X_train), "  validation:", len(X_val), "  test:", len(X_test))

### Step 7: Scale, then make tensors

**Slide reminder:** `StandardScaler` puts every input on the **same size**. Fit it on the **train** data only (we don't peek at the exam!).

In [ ]:
scaler = StandardScaler().fit(X_train)

def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)

X_train_t = to_tensor(scaler.transform(X_train))
X_val_t   = to_tensor(scaler.transform(X_val))
X_test_t  = to_tensor(scaler.transform(X_test))
y_train_t = to_tensor(y_train).reshape(-1, 1)      # one column
y_val_t   = to_tensor(y_val).reshape(-1, 1)

### Step 8: The training loop

**Slide reminder:** always the same **5 lines**, repeated for every **epoch**:

1. guess → 2. how wrong → 3. clear old blame → 4. find blame → 5. fix

After every epoch we also write down the **validation loss** (the mock exam). `Adam` = a smarter optimizer that picks the step size by itself.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1))    # a fresh map, random weights
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

train_losses, val_losses = [], []
for epoch in range(300):
    guess = model(X_train_t)                 # 1. guess (forward pass)
    loss = loss_fn(guess, y_train_t)         # 2. how wrong (loss)
    optimizer.zero_grad()                    # 3. clear old blame
    loss.backward()                          # 4. find blame (backprop)
    optimizer.step()                         # 5. fix (gradient descent)
    train_losses.append(loss.item())

    with torch.no_grad():                    # mock exam: only check, no learning
        val_losses.append(loss_fn(model(X_val_t), y_val_t).item())

    if epoch % 50 == 0:
        print(f"epoch {epoch:3}:  train loss = {train_losses[-1]:7.1f}   validation loss = {val_losses[-1]:7.1f}")

The loss starts **huge** (random weights) and goes **down**, epoch after epoch. The network is learning.

### Step 9: Is training going well? Read the loss curves

**Slide reminder:**

| What you see | Meaning |
|--------------|---------|
| both go down, and stay close | ✓ learning well |
| train goes down, validation goes **up** | ✗ overfitting: stop training sooner |
| both jump up and down | ✗ steps too big: smaller learning rate |

In [ ]:
plt.plot(train_losses, label="train (homework)")
plt.plot(val_losses, label="validation (mock exam)")
plt.ylim(0, 1000)                  # the first epochs are huge; zoom in on the rest
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

Both lines go down and **stay close**: ✓ learning well, no overfitting.

### Step 10: The final exam, and compare with a simple model

**Slide reminder:** neural networks are not magic. **Always compare with a simple model.**

**MAE** = on average, how many marks the guess is wrong.

In [ ]:
with torch.no_grad():
    nn_guess = model(X_test_t).numpy().ravel()          # .ravel() = one flat list
nn_mae = mean_absolute_error(y_test, nn_guess)

lin = LinearRegression().fit(X_train, y_train)          # the simple model, same homework
lin_mae = mean_absolute_error(y_test, lin.predict(X_test))

print("Neural network:    wrong by", round(nn_mae, 1), "marks on average")
print("Linear Regression: wrong by", round(lin_mae, 1), "marks on average")

Both are **about the same**. Marks here follow a **simple** pattern, so a simple model is enough.
Neural networks win on **harder** data: photos (CNN), sentences (RNN, Transformer).

### Step 11: Use the trained network

Back to our student from the slides: **3 study hours, 2 phone hours**. Before training, the guess was **8.7**. Now?

(New data must be **scaled with the same scaler**.)

In [ ]:
new_student = [[3, 2]]
with torch.no_grad():
    guess = model(to_tensor(scaler.transform(new_student)))
print("Guess for 3 study hours, 2 phone hours:", round(guess.item(), 1), "marks")

A sensible guess now: about **62 marks**, instead of 8.7. Same network, same 13 numbers. Only the **weights and biases changed**: they were **learned** from the data.

### ✏️ QUICK CHECK 3: Ask the network

Guess the mark for:
1. a student who studies **6 hours** and uses the phone **1 hour**
2. a student who studies **1 hour** and uses the phone **5 hours**

Does the answer make sense?

In [ ]:
# Write your code here

## The complete program

Everything from Part B in **one cell**. This is what a real neural network project looks like.

In [ ]:
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. inputs and real answers
students = pd.read_csv("students_100.csv")
X = students[["study_hours", "phone_hours"]].values
y = students["maths"].values

# 2. split: homework 60% / mock exam 20% / final exam 20%
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)

# 3. scale (fit on train only) + tensors
scaler = StandardScaler().fit(X_train)
def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)
X_train_t, X_val_t, X_test_t = (to_tensor(scaler.transform(d)) for d in (X_train, X_val, X_test))
y_train_t, y_val_t = to_tensor(y_train).reshape(-1, 1), to_tensor(y_val).reshape(-1, 1)

# 4. the map: 2 inputs → 3 hidden (ReLU) → 1 output
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1))
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# 5. training loop: guess → how wrong → clear → blame → fix
for epoch in range(300):
    loss = loss_fn(model(X_train_t), y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 6. mock exam
with torch.no_grad():
    print("Validation loss:", round(loss_fn(model(X_val_t), y_val_t).item()))

# 7. final exam + compare with the simple model
with torch.no_grad():
    nn_mae = mean_absolute_error(y_test, model(X_test_t).numpy().ravel())
lin_mae = mean_absolute_error(y_test, LinearRegression().fit(X_train, y_train).predict(X_test))
print("Neural network MAE:   ", round(nn_mae, 1))
print("Linear Regression MAE:", round(lin_mae, 1))

# 8. use it
with torch.no_grad():
    print("3 study hours, 2 phone hours →", round(model(to_tensor(scaler.transform([[3, 2]]))).item(), 1), "marks")

# 🧑‍🏫 Example 2: A network that reads handwritten digits

**The problem:** the post office wants a computer to read **handwritten numbers** (0 to 9) on letters.

Nobody can write rules for "this is a 7". So we give a neural network **many examples** of handwritten digits **with the answers**, and it **finds the rules by itself**. (Slide: *What is a neural network?*)

**What is new compared to Example 1:**

| | Example 1: Maths marks | Example 2: Digits |
|---|---|---|
| Inputs | 2 numbers (study, phone) | **64 numbers** (the pixels of a small picture) |
| Answer | one number (a mark) | **one of 10 classes** (0, 1, 2 … 9) |
| Output neurons | 1 | **10** (one score for each digit) |
| Loss | `nn.MSELoss` | `nn.CrossEntropyLoss` (for choosing a class) |

Everything else is **the same**: layers, ReLU, the 5-line training loop, train / validation / test.

### Step 1: Load the pictures

The data comes **inside sklearn**, no file needed: **1,797** small pictures of handwritten digits.

In [ ]:
from sklearn.datasets import load_digits     # a small built-in dataset of handwritten digits

digits = load_digits()                       # load all the pictures and their answers
print("Pictures:", digits.images.shape)      # (1797, 8, 8) = 1797 pictures, each 8 × 8 pixels
print("Answers: ", digits.target[:10])       # the real digit for the first 10 pictures

### Step 2: What does the computer see?

**Slide reminder (Inputs):** a computer only understands **numbers**.

A picture is just a **grid of numbers**: one number per pixel. **0 = white** (empty), **16 = black** (full ink).

In [ ]:
first = digits.images[0]                     # the first picture: an 8 × 8 grid of numbers
print("The answer is:", digits.target[0])    # which digit it really is
print(first.astype(int))                     # print the grid as whole numbers (0 = empty, 16 = full ink)

Can you see the shape of a **0** in the numbers? The big numbers make a ring.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 2))            # 10 small pictures in one row
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray_r")              # draw the grid of numbers as a picture
    ax.set_title(digits.target[i])                          # the real answer on top
    ax.axis("off")                                          # hide the axis numbers
plt.show()

### Step 3: The inputs: 64 numbers in one row

A network wants **one row per example**. So we put the 8 rows of the picture **one after another**: 8 × 8 = **64 numbers**.

`digits.data` is already like that. We also **divide by 16**, so every pixel is between **0 and 1** (the same size, like scaling).

In [ ]:
X = digits.data / 16          # 64 pixels per picture, now between 0 and 1
y = digits.target             # the real digit (0 to 9) for each picture
print("X:", X.shape, "  y:", y.shape)    # (1797, 64) = 1797 pictures, 64 inputs each

### Step 4: Split into train / validation / test

Same as Example 1: homework 60%, mock exam 20%, final exam 20%.

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)            # take out the final exam
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42) # split the rest: homework + mock exam
print("train:", len(X_train), "  validation:", len(X_val), "  test:", len(X_test))

X_train_t = torch.tensor(X_train, dtype=torch.float32)   # pixels: decimal numbers
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)      # answers: whole numbers (class 0 to 9), so we use "long"
y_val_t   = torch.tensor(y_val,   dtype=torch.long)

### Step 5: Build the map for digits

```
   64 inputs (pixels)  ──▶  32 hidden neurons (ReLU)  ──▶  10 outputs: a score for 0, 1, 2 … 9
```

**Why 10 outputs?** One neuron for **each digit**. Each one gives a **score**: "how much does this picture look like **my** digit?"
The digit with the **biggest score wins**. Like 10 judges: each judge only checks one digit.

In [ ]:
torch.manual_seed(42)              # the same random start for everyone
digit_model = nn.Sequential(
    nn.Linear(64, 32),             # 64 pixels → 32 hidden neurons (weights + biases)
    nn.ReLU(),                     # activation: negative → 0
    nn.Linear(32, 10),             # 32 hidden neurons → 10 scores, one for each digit
)
print(digit_model)
print("Weights + biases to learn:", sum(p.numel() for p in digit_model.parameters()))   # 64×32+32 + 32×10+10

**2,410 numbers to learn!** Example 1 had only 13. More inputs and more neurons → many more weights.

### Step 6: Ask the network BEFORE training

The weights are random. Let's see what it guesses for the first test picture.

In [ ]:
with torch.no_grad():                                   # only guessing, no learning
    scores = digit_model(X_test_t[:1])                  # forward pass for 1 picture → 10 scores
print("10 scores:", scores.round(decimals=2))           # one score per digit 0 … 9
print("Guess:", scores.argmax().item(), "  Real:", y_test[0])   # argmax = the position of the biggest score

A **random** guess. The network knows nothing yet. (`argmax` = *which position has the biggest number?*)

### Step 7: The loss for choosing a class

**Slide reminder (Loss):** one number for "how wrong".

For classes we use `nn.CrossEntropyLoss`:
- the network gives the **right digit** a **big** score → **small** loss ✓
- the network gives the right digit a **small** score → **big** loss ✗

In [ ]:
loss_fn = nn.CrossEntropyLoss()                               # the loss for choosing between classes
print("Loss before training:", round(loss_fn(digit_model(X_train_t), y_train_t).item(), 2))

### Step 8: Train it: the same 5 lines

We also measure **accuracy** = how many pictures it reads **correctly**, in %.

In [ ]:
optimizer = torch.optim.Adam(digit_model.parameters(), lr=0.01)    # Adam: picks the step size by itself

def accuracy(model, X_t, y_true):
    with torch.no_grad():                                          # only checking, no learning
        guesses = model(X_t).argmax(dim=1).numpy()                 # biggest score = the guess, for every picture
    return (guesses == y_true).mean() * 100                        # % of correct guesses

train_losses, val_losses = [], []
for epoch in range(100):
    scores = digit_model(X_train_t)              # 1. guess: 10 scores for every picture
    loss = loss_fn(scores, y_train_t)            # 2. how wrong
    optimizer.zero_grad()                        # 3. clear old blame
    loss.backward()                              # 4. find blame (backprop) for all 2,410 weights
    optimizer.step()                             # 5. fix: every weight takes a small step
    train_losses.append(loss.item())

    with torch.no_grad():                        # mock exam: check, no learning
        val_losses.append(loss_fn(digit_model(X_val_t), y_val_t).item())

    if epoch % 20 == 0:
        print(f"epoch {epoch:3}:  loss = {loss.item():.3f}   mock exam accuracy = {accuracy(digit_model, X_val_t, y_val):.1f}%")

At epoch 0 it is right only about **1 time in 4**. After 60 epochs: **more than 95%**! The loss goes down, the accuracy goes up.

In [ ]:
plt.plot(train_losses, label="train (homework)")      # loss on the homework
plt.plot(val_losses, label="validation (mock exam)")  # loss on the mock exam
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

Both lines go down and stay close: ✓ learning well.

### Step 9: The final exam

The test pictures were **never** seen during training.

In [ ]:
test_acc = accuracy(digit_model, X_test_t, y_test)            # % correct on the final exam
print("Final exam accuracy:", round(test_acc, 1), "%")
print("Correct:", int(round(test_acc / 100 * len(y_test))), "out of", len(y_test), "pictures")

### Step 10: Look at its answers

Green title = correct. Red title = a mistake.

In [ ]:
with torch.no_grad():
    guesses = digit_model(X_test_t).argmax(dim=1).numpy()    # the network's guess for every test picture

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))            # 16 test pictures
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")        # 64 numbers back into an 8 × 8 picture
    color = "green" if guesses[i] == y_test[i] else "red"    # green = right, red = wrong
    ax.set_title(f"guess {guesses[i]}", color=color)
    ax.axis("off")
plt.show()

And the **mistakes**: are they hard for a human too?

In [ ]:
wrong = (guesses != y_test).nonzero()[0][:8]                  # the positions of the first 8 mistakes
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 2))
for ax, i in zip(axes, wrong):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"guess {guesses[i]}\nreal {y_test[i]}", color="red")   # what it said vs the truth
    ax.axis("off")
plt.show()

Some of these are **messy** even for us. The pictures are only 8 × 8 pixels!

### Step 11: How sure is the network?

`torch.softmax` turns the 10 scores into **10 chances that add up to 100%**. (Like sigmoid from the slides, but for 10 choices.)

In [ ]:
with torch.no_grad():
    chances = torch.softmax(digit_model(X_test_t[:1]), dim=1)[0] * 100   # 10 chances in %, for the first test picture

print("Real digit:", y_test[0])
for digit, chance in enumerate(chances):
    print(f"  {digit}: {chance:5.1f}%  " + "█" * int(chance // 5))     # a small bar: one block = 5%

### ✏️ QUICK CHECK 4: Test a picture yourself

1. Choose a test picture: `i = 25` (then try other numbers).
2. Draw it with `plt.imshow(X_test[i].reshape(8, 8), cmap="gray_r")`.
3. Print the network's guess, the real digit, and how sure it is (the biggest chance).

In [ ]:
# Write your code here

**What we learned:**
> The network was never told what a "7" looks like. It saw **1,000+ examples with answers**, and changed its **2,410 weights and biases** until it could read digits by itself. Your phone reading a cheque, or Google Lens reading text, works the same way, with **bigger** pictures and a **CNN** (slide: *Types of neural networks*).

## The complete program: digits

Everything from Example 2 in **one cell**.

In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# 1. inputs (64 pixels, 0..1) and answers (0..9)
digits = load_digits()
X, y = digits.data / 16, digits.target

# 2. split: homework 60% / mock exam 20% / final exam 20%
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)
X_train_t, X_val_t, X_test_t = (torch.tensor(d, dtype=torch.float32) for d in (X_train, X_val, X_test))
y_train_t = torch.tensor(y_train, dtype=torch.long)

# 3. the map: 64 pixels → 32 hidden (ReLU) → 10 scores
torch.manual_seed(42)
digit_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(digit_model.parameters(), lr=0.01)

# 4. training loop: guess → how wrong → clear → blame → fix
for epoch in range(100):
    loss = loss_fn(digit_model(X_train_t), y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 5. mock exam + final exam
with torch.no_grad():
    val_acc = (digit_model(X_val_t).argmax(dim=1).numpy() == y_val).mean() * 100
    test_acc = (digit_model(X_test_t).argmax(dim=1).numpy() == y_test).mean() * 100
print("Mock exam accuracy: ", round(val_acc, 1), "%")
print("Final exam accuracy:", round(test_acc, 1), "%")

# 🏠 Home Problem: guess the price of a used car

A used-car showroom wants to guess a car's **price** from 3 facts. **Same steps as Part B**, new data.

The data is in `cars.csv`:

| Column | Meaning |
|--------|---------|
| `car_age` | age of the car in years (input) |
| `km_thousands` | how much it has been driven, in 1000 km (input) |
| `engine_cc` | engine size (input) |
| `price_lakh` | the price in lakh rupees (**the answer**) |

**Draw the map first** (on paper): how many inputs? How many weights + biases with 3 hidden neurons? *(Answer: 3 inputs. 3 × 3 + 3 = 12, plus 3 × 1 + 1 = 4 → 16.)*

### Step 1: Load the data

Make `X` (the 3 input columns) and `y` (`price_lakh`), with `.values`. Print the shapes.

In [ ]:
# Write your code here

### Step 2: Split into train / validation / test (60 / 20 / 20)

Use `random_state=42`, like the class problem. Print the sizes.

In [ ]:
# Write your code here

### Step 3: Scale, then make tensors

Why is scaling **very** important here? Look at the numbers: `car_age` is 0 to 15, but `engine_cc` is 660 to 1800!

In [ ]:
# Write your code here

### Step 4: Build the network

**3 inputs** → **3 hidden neurons** (ReLU) → **1 output**. Print the number of weights + biases. Is it 16?

In [ ]:
# Write your code here

### Step 5: Train it

Use `nn.MSELoss()`, `Adam` with `lr=0.1`, and **300 epochs**. Write the **5 lines** yourself. Save the train and validation losses.

In [ ]:
# Write your code here

### Step 6: Draw the loss curves

Which shape is it: learning well, overfitting, or steps too big?

In [ ]:
# Write your code here

### Step 7: Final exam: compare with Linear Regression

Calculate the test MAE of both models. Which one would you choose for the showroom, and why?

In [ ]:
# Write your code here

### Step 8: Use it

Guess the price of a **5-year-old** car, driven **60 thousand km**, with a **1300 cc** engine.
Then try a **1-year-old**, **10 thousand km**, **1800 cc** car. Which is more expensive? Does it make sense?

In [ ]:
# Write your code here

**Bring to the next class:** your notebook, your loss curve, and your answer to the showroom (Step 7).

# Key Takeaways

- `nn.Linear` = the **lines (weights)** and **+b (biases)** of the map. `nn.ReLU` = the activation.
- `nn.Sequential` puts the layers **in a row**: the **forward pass** goes left → right.
- **Loss** (`nn.MSELoss`) = how wrong. **Backprop** (`loss.backward()`) = find the blame. **Gradient descent** (`optimizer.step()`) = fix.
- A **too big** learning rate makes the loss **explode**.
- Training loop: **guess → how wrong → clear → blame → fix**, for many **epochs**.
- **Train / validation / test** = homework / mock exam / final exam.
- **Digits:** 64 pixel inputs → **10 outputs** (one score per digit), `nn.CrossEntropyLoss`, the biggest score wins.
- Before training: a random guess (8.7). After training: a sensible guess (about 62). **Learning = changing the weights and biases.**